# 🏏 IPL Win Probability Predictor
In this project, we predict the win probability of the chasing team in the 2nd innings of an IPL match.

### 📋 Custom Preprocessing Requirements:
1. **Replace Team Names:**
   - Delhi Daredevils ➡️ Delhi Capitals
   - Deccan Chargers ➡️ Sunrisers Hyderabad
   - Kings XI Punjab ➡️ Punjab Kings
   - Royal Challengers Bangalore ➡️ Royal Challengers Bengaluru
2. **Remove Defunct Teams:**
   - Gujarat Lions
   - Rising Pune Supergiant / Rising Pune Supergiants
   - Kochi Tuskers Kerala
   - Pune Warriors
3. **Train Machine Learning Model (Logistic Regression Pipeline with OneHotEncoder)**
4. **Save Pipeline as pipe.pkl for the Streamlit Web Application**


In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report


## 1. Load Dataset

In [ ]:
df = pd.read_csv('matches.csv')
print('Dataset Shape:', df.shape)
df.head()


## 2. Step 1: Team Name Replacements
Here we replace old franchise names with their modern counterparts:
- **Delhi Daredevils** ➡️ **Delhi Capitals**
- **Deccan Chargers** ➡️ **Sunrisers Hyderabad**
- **Kings XI Punjab** ➡️ **Punjab Kings**
- **Royal Challengers Bangalore** ➡️ **Royal Challengers Bengaluru**


In [ ]:
team_replacements = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

df['batting_team'] = df['batting_team'].replace(team_replacements)
df['bowling_team'] = df['bowling_team'].replace(team_replacements)

print('Teams after replacement:')
print(df['batting_team'].unique())


## 3. Step 2: Removing Inactive/Defunct Teams
Remove matches involving teams that no longer play in IPL:
- Gujarat Lions
- Rising Pune Supergiant / Rising Pune Supergiants
- Kochi Tuskers Kerala
- Pune Warriors


In [ ]:
teams_to_remove = [
    'Gujarat Lions', 
    'Rising Pune Supergiant', 
    'Rising Pune Supergiants', 
    'Kochi Tuskers Kerala', 
    'Kochi Tkers Kerala',
    'Pune Warriors'
]

# Filter rows where either batting or bowling team is obsolete
df = df[~df['batting_team'].isin(teams_to_remove)]
df = df[~df['bowling_team'].isin(teams_to_remove)]

print('Filtered Shape:', df.shape)
active_teams = sorted(df['batting_team'].unique())
print(f'Active IPL Teams ({len(active_teams)}):\n', active_teams)


## 4. Step 3: Venue Standardization
Group stadium duplicate variations into clean, recognizable names.


In [ ]:
def clean_venue(v):
    v_low = str(v).lower().strip()
    if 'wankhede' in v_low:
        return 'Wankhede Stadium, Mumbai'
    elif 'chinnaswamy' in v_low:
        return 'M Chinnaswamy Stadium, Bengaluru'
    elif 'eden gardens' in v_low:
        return 'Eden Gardens, Kolkata'
    elif 'chidambaram' in v_low or 'chepauk' in v_low:
        return 'MA Chidambaram Stadium, Chennai'
    elif 'kotla' in v_low or 'arun jaitley' in v_low:
        return 'Arun Jaitley Stadium, Delhi'
    elif 'rajiv gandhi' in v_low or 'uppal' in v_low:
        return 'Rajiv Gandhi Stadium, Hyderabad'
    elif 'punjab cricket' in v_low or 'bindra' in v_low or 'mohali' in v_low:
        return 'PCA Stadium, Mohali'
    elif 'sawai mansingh' in v_low or 'jaipur' in v_low:
        return 'Sawai Mansingh Stadium, Jaipur'
    elif 'narendra modi' in v_low or 'motera' in v_low or 'sardar patel' in v_low:
        return 'Narendra Modi Stadium, Ahmedabad'
    elif 'ekana' in v_low or 'lucknow' in v_low:
        return 'BRSABV Ekana Stadium, Lucknow'
    elif 'dy patil' in v_low:
        return 'DY Patil Stadium, Mumbai'
    elif 'brabourne' in v_low:
        return 'Brabourne Stadium, Mumbai'
    elif 'dubai' in v_low:
        return 'Dubai International Cricket Stadium'
    elif 'sharjah' in v_low:
        return 'Sharjah Cricket Stadium'
    elif 'zayed' in v_low:
        return 'Sheikh Zayed Stadium, Abu Dhabi'
    elif 'dharamsala' in v_low or 'himachal' in v_low:
        return 'HPCA Stadium, Dharamsala'
    elif 'visakhapatnam' in v_low or 'rajasekhara' in v_low:
        return 'ACA-VDCA Stadium, Visakhapatnam'
    elif 'mca' in v_low or 'maharashtra cricket' in v_low or 'pune' in v_low:
        return 'MCA Stadium, Pune'
    return v.strip()

df['venue'] = df['venue'].apply(clean_venue)
print('Unique Cleaned Venues:', df['venue'].nunique())


## 5. Step 4: Sanity Checks & Cleaning

In [ ]:
# Remove zero balls left or invalid required run rates
df = df[df['balls_left'] > 0]
df = df[df['target'] > 0]
df = df[(df['required_run_rate'] >= 0) & (df['required_run_rate'] < 50)]
df = df[df['wickets_left'] >= 0]
df = df[df['runs_left'] >= 0]
print('Final Cleaned Dataset Shape:', df.shape)


## 6. Step 5: Feature Selection & Model Training

In [ ]:
features = [
    'batting_team', 
    'bowling_team', 
    'venue', 
    'runs_left', 
    'balls_left', 
    'wickets_left', 
    'target', 
    'current_run_rate', 
    'required_run_rate'
]

X = df[features]
y = df['batting_team_won']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline with OneHotEncoder + LogisticRegression
trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop=None, handle_unknown='ignore'), ['batting_team', 'bowling_team', 'venue'])
], remainder='passthrough')

pipe = Pipeline([
    ('step1', trf),
    ('step2', LogisticRegression(solver='liblinear', random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%')


## 7. Step 6: Save Model as pipe.pkl

In [ ]:
with open('pipe.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print('Model successfully saved to pipe.pkl!')
